# SRQ generalization M4 — RanPAC train-only gate

This notebook evaluates the official RanPAC Phase-2 random-ReLU analytic head with Exact Gram, FP32/FP16 square-root, and frozen P2B INT8 backends. It never materializes CIFAR-100 test features.

In [ ]:
# Edit repository/path values only. Protocol, selection, and gates are source-locked.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m4_cifar_features'
OUTPUT_DIR='/content/srq_m4_ranpac_output'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU check, and immutable source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED={
 'configs/srq_generalization_m4_ranpac_train_only.json':'8867bf87c9c2c83ef81dfccac668b63d53fa67bac806bc979f827b1a57fcd53f',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'methods/analytic_ridge/backends.py':'40114dcc05a9d991682f840efc48e7c74a3e12dde99be049a30ad5f825650dee',
 'methods/frontends/ranpac.py':'6b94532f607d245c0d148d09c1a36b44dbbf964f4ee9cbf05ba6f64826a90e60'}
for path,expected in EXPECTED.items(): assert sha(path)==expected,(path,sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
CONFIG='configs/srq_generalization_m4_ranpac_train_only.json'
RUNNER='tools/srq_generalization_m4.py'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('SOURCE LOCK: PASS')

In [ ]:
# Fast local gates before downloading data.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_generalization_m4.py','tests/test_ranpac_analytic_frontend.py','tests/test_analytic_ridge_backend.py','tests/test_analytic_ridge_equivalence.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M4 synthetic correctness gate failed; return the complete traceback.'
print('M4 SYNTHETIC GATE: PASS')

In [ ]:
# Download the locked backbone checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out test features remain absent.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m4','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Run fixed-lambda calibration and all five analytic paths on one train-only stream.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M4 START: source-locked RanPAC head, 10 tasks, five analytic paths.',flush=True)
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'m4_results.json'
assert result_path.is_file(),'M4 failed before writing diagnostics; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='PASS_M4_RANPAC_TRAIN_ONLY','M4 failed; do not relax gates or inspect test accuracy.'

In [ ]:
# Human-readable per-task audit.
import pandas as pd
rows=[]
for record in result['records']:
    row={'task':record['task'],'fp32_system_error':record['fp32_system_relative_error'],'fp32_prediction_agreement':record['evaluation']['reference_fp32_prediction_agreement']}
    row.update({name+'_accuracy':value for name,value in record['evaluation']['accuracy_percent'].items()})
    rows.append(row)
display(pd.DataFrame(rows))

In [ ]:
# Export evidence only; the sample-level feature cache is excluded.
bundle=Path('/content/srq_generalization_m4_ranpac_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(Path(OUTPUT_DIR)/'m4_results.json',bundle/'m4_results.json')
shutil.copy2(CONFIG,bundle/'config.json')
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha(archive))
from google.colab import files
files.download(archive)